# AI-Based Pothole Recognition Using UAVs (Version 2.0)
## Google Colab 50-Epoch GPU Training Notebook (YOLOv11s SOTA)

**Project:** Automated UAV Road Inspection & Municipal Priority Dispatch  
**Course:** BCSE306L – Artificial Intelligence (VIT)  
**Faculty:** Dr. Vijayaprabakaran K  
**Students:** Ajay Kumaar (24BRS1287) & Hariaswath (24BRS1290)  

---
### Version 2.0 Enhancements
1. **Negative Background Samples**: Includes clean pavement, tree canopies, and road shadow images with 0 annotations to suppress false positives.
2. **Color-Preserving Augmentation (`hsv_h = 0.0`)**: Prevents green trees from turning gray during training so the network explicitly learns that Green Foliage $\neq$ Asphalt Potholes.
3. **YOLOv11s Architecture**: Uses YOLOv11 Small (9.4M parameters) with C2PSA spatial attention for small defect feature extraction.

---
### Instructions
1. In Google Colab, go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Drag and drop `pothole_dataset_v2.zip` into the Colab file explorer on the left.
3. Run all cells. In ~5 minutes, it trains 50 epochs and automatically downloads your trained `best.pt` model!

In [ ]:
# Step 1: Verify Free T4 GPU Acceleration
!nvidia-smi

# Step 2: Install Ultralytics and dependencies
!pip install -q ultralytics opencv-python pillow matplotlib pyyaml tqdm

# Step 3: Unzip the uploaded dataset
!unzip -q -o pothole_dataset_v2.zip
print("Dataset V2 successfully unzipped!")

In [ ]:
# Step 4: Verify PyTorch CUDA Support
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Accelerated GPU Device: {torch.cuda.get_device_name(0)}")

---
### Step 5: Train YOLOv11s for 50 Epochs (Color-Preserving SOTA)
Trains YOLOv11s on 640x640 images with `hsv_h = 0.0` to preserve vegetation colors.

In [ ]:
from ultralytics import YOLO

# Initialize YOLOv11s (Small) model
model_yolo11s = YOLO('yolo11s.pt')

# Train with color-preserving hyperparameter config
results_yolo11s = model_yolo11s.train(
    data='dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    hsv_h=0.0,       # Disable hue shift to preserve green tree vegetation integrity
    hsv_s=0.5,       # Saturation augmentation
    hsv_v=0.4,       # Value/Luminance augmentation for shadow robustness
    degrees=10.0,    # Rotation augmentation
    translate=0.1,   # Translation
    scale=0.5,       # Multiscale training
    mosaic=1.0,      # Mosaic augmentation
    device=0 if torch.cuda.is_available() else 'cpu',
    project='runs/colab',
    name='yolo11s_sota_v2',
    plots=True,
    verbose=True
)

---
### Step 6: Validate Precision, Recall & mAP Metrics

In [ ]:
val_results = model_yolo11s.val()
print("=" * 60)
print("FINAL 50-EPOCH MODEL V2 EVALUATION METRICS:")
print(f"Precision:  {val_results.results_dict.get('metrics/precision(B)', 0):.4f}")
print(f"Recall:     {val_results.results_dict.get('metrics/recall(B)', 0):.4f}")
print(f"mAP@50:     {val_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP@50-95:  {val_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
print("=" * 60)

---
### Step 7: Download the Trained Checkpoint (`best.pt`)
This automatically triggers a browser download for the trained `best.pt` weights file.

In [ ]:
from google.colab import files
print("Downloading best.pt to your computer...")
files.download('runs/colab/yolo11s_sota_v2/weights/best.pt')